Импорты

In [39]:
import pandas as pd
import numpy as np
import json



Загрузка данных

In [40]:
pairs = pd.read_parquet('data/pairs.parquet', columns=['id1', 'id2', 'target', 'attributes_1', 'attributes_2'])

In [41]:
pairs.head()

,id1,id2,target,attributes_1,attributes_2
0,32,188978721454,0.0,"{""диоптрии"":""- 2.00"",""цвет оправы"":""золотистый...","{""наличие флексов"":""да"",""затемнение"":""коричнев..."
1,34,197568543106,0.0,"{""бренд"":""r.o.c.s."",""тип"":""зубная щетка"",""уров...","{""целевая аудитория"":""взрослая"",""бренд"":""r.o.c..."
2,37,661425040668,0.0,"{""бренд в одежде и обуви"":""fre gamo"",""страна б...","{""цвет"":""желтый"",""страна производства"":""китай""..."
3,97,274877917548,0.0,"{""ось (axis)"":""10"",""бренд"":""acuvue"",""цилиндр (...","{""состав"":""силикон-гидрогель"",""тип линз"":""тори..."
4,154,283467899529,0.0,"{""бренд"":""weiserhouse"",""условия хранения"":""хра...","{""бренд"":""чайная линия"",""вид чая"":""пуэр"",""вкус..."


In [42]:
no_article = ['нет', '-', '0', 'null', '_']
no_brand = ['нет бренда', 'не определен', 'noname', 'nobrand', 'без бренда', 'нет', 'нет бренд', '---','неизвестен']

def clean(value, bad_values, min_len=1):
    if value is None:
        return None
    value = value.lower().strip()
    if value == '' or value in bad_values or len(value) < min_len:
        return None
    return value



def parse_attributes(attribute):
    attribute = json.loads(attribute)
    brand_1 = attribute.get('бренд')
    brand_2 = attribute.get('бренд в одежде и обуви')
    article = attribute.get('артикул')
    brand_clean = clean(brand_1, no_brand)

    if brand_clean is None:
        brand_clean = clean(brand_2, no_brand)
    article_clean = clean(article, no_article, 3)
    partnumbers = ['артикул производителя', 'партномер (артикул производителя)', 'партномер']

    for partnumber in partnumbers:
        clean_partnumber= clean(attribute.get(partnumber), no_article, 3)
        if clean_partnumber is not None:
            break

    return brand_clean, article_clean, clean_partnumber


In [43]:
parsed_1 = pairs['attributes_1'].apply(parse_attributes).tolist()
df = pd.DataFrame(parsed_1, columns=['brand', 'article', 'partnumber'])
pairs['brand_1'] = df['brand']
pairs['article_1'] = df['article']
pairs['partnumber_1'] = df['partnumber']

parsed_2 = pairs['attributes_2'].apply(parse_attributes).tolist()
df = pd.DataFrame(parsed_2, columns=['brand', 'article', 'partnumber'])
pairs['brand_2'] = df['brand']
pairs['article_2'] = df['article']
pairs['partnumber_2'] = df['partnumber']

In [44]:
print(pairs.head())

   id1           id2  target  \
0   32  188978721454     0.0   
1   34  197568543106     0.0   
2   37  661425040668     0.0   
3   97  274877917548     0.0   
4  154  283467899529     0.0   

                                        attributes_1  \
0  {"диоптрии":"- 2.00","цвет оправы":"золотистый...   
1  {"бренд":"r.o.c.s.","тип":"зубная щетка","уров...   
2  {"бренд в одежде и обуви":"fre gamo","страна б...   
3  {"ось (axis)":"10","бренд":"acuvue","цилиндр (...   
4  {"бренд":"weiserhouse","условия хранения":"хра...   

                                        attributes_2      brand_1 article_1  \
0  {"наличие флексов":"да","затемнение":"коричнев...         None      None   
1  {"целевая аудитория":"взрослая","бренд":"r.o.c...     r.o.c.s.      None   
2  {"цвет":"желтый","страна производства":"китай"...     fre gamo      None   
3  {"состав":"силикон-гидрогель","тип линз":"тори...       acuvue      None   
4  {"бренд":"чайная линия","вид чая":"пуэр","вкус...  weiserhouse      None

In [45]:
pairs = pairs[['id1', 'id2', 'target',
               'brand_1', 'brand_2',
               'article_1', 'article_2',
               'partnumber_1', 'partnumber_2']]

In [46]:
print(pairs)

                 id1           id2  target      brand_1       brand_2  \
0                 32  188978721454     0.0         None        fedrov   
1                 34  197568543106     0.0     r.o.c.s.      r.o.c.s.   
2                 37  661425040668     0.0     fre gamo          None   
3                 97  274877917548     0.0       acuvue          None   
4                154  283467899529     0.0  weiserhouse  чайная линия   
...              ...           ...     ...          ...           ...   
365649  850403683693  695784731688     0.0        alcon          None   
365650  850403683737  369367295238     0.0         None      carp box   
365651  850403683764  377957271291     1.0         lori          None   
365652  850403683771  231928288442     0.0         None          None   
365653  850403683774  171798796415     0.0         None          None   

       article_1 article_2 partnumber_1 partnumber_2  
0           None      None         None         None  
1           N

In [64]:
print(pairs.notna().mean())

id1             1.000000
id2             1.000000
target          1.000000
brand_1         0.491046
brand_2         0.490644
article_1       0.132738
article_2       0.133993
partnumber_1    0.093616
partnumber_2    0.093744
dtype: float64


In [65]:
pairs.to_parquet('data/pairs_attrs.parquet')